# Shallow Water Layer (single layer)

This is the canonical single-layer of constant density, incompressible fluid.

## Equations

For a single layer, the governing equations are
\begin{align}
\partial_t h + \partial_x \left( h u \right) + \partial_y \left( h v \right) & = 0 \\
\partial_t u - q h v + \partial_x \left( M + K \right) & = \frac{\tau^x - \epsilon u}{D} + \nu \nabla^2 u \\
\partial_t v + q h u + \partial_y \left( M + K \right) & = \frac{\tau^y - \epsilon v}{D} + \nu \nabla^2 v
\end{align}
where
\begin{align}
\eta & = -D + h \\
M & = g \eta \\
K & = \frac{1}{2} \left( u^2 + v^2 \right) \\
q & = \frac{ f + \zeta }{ h } \\
\zeta & = \partial_x v - \partial_y u \\
f & = f_o + \beta y
\end{align}

## Algorithm

$u$, $v$ and $h$ will be staggered on an Arakawa C-grid using south-west indexing convention, i.e. $u_{i-1/2,j}=u[j,i]$ and $v_{i,j-1/2}=v[j,i]$ are west and south of $h_{i,j}=h[j,i]$ respectively.

Time discretization and algorithm (only partial spatial discretization so far, $\tilde{ \cdot }$ indicates upwinding of some form):
\begin{align}
H^u & = \tilde{ h }^i u^n \\
h^* &= h^n - \frac{\Delta t}{\Delta x} \delta_i H^u \\
H^v & = \tilde{ h^* }^j v^n \\
h^{n+1} &= h^* - \frac{\Delta t}{\Delta y} \delta_j H^v \\
K_{i,j}^n & = \max(0,u_{i-1/2,j}^n)^2 + \min(0,u_{i+1/2,j}^n)^2 + \max(0,v_{i,j-1/2}^n)^2 + \min(0,v_{i,j+1/2}^n)^2 \\
q & = \frac{ f + \delta_i ( v^n \Delta y ) - \delta_j ( u^n \Delta x ) }{ h^{n} \Delta x \Delta y } \\
\eta^{n+1} &= -D + h^{n+1} \\
M^{n+1} &= g \eta^{n+1}
\end{align}
and a simultaneous solve of
\begin{align}
\frac{ u^{n+1} - u^n }{ \Delta t}
& = f \left( \alpha_f v^{n+1} + ( 1 - \alpha_f ) v^n \right) +
\left( \tilde{q}^{j} \overline{ H^v }^{ij} - f v^n \right) - \frac{1}{\Delta x} \delta_i \left( M^{n+1} + K^n \right)
+ \frac{\tau^x}{D} - \frac{\epsilon \left( \alpha_\epsilon u^{n+1} + ( 1 - \alpha_\epsilon ) u^n \right)}{D} + \nu \nabla^2 u^n \\
\frac{ v^{n+1} - v^n }{ \Delta t}
& = - f \left( \alpha_f u^{n+1} + ( 1 - \alpha_f ) u^n \right) +
\left( - \tilde{q}^{i} \overline{ H^u }^{ij} + f u^n \right) - \frac{1}{\Delta y} \delta_j \left( M^{n+1} + K^n \right)
+ \frac{\tau^y}{D} - \frac{\epsilon \left( \alpha_\epsilon v^{n+1} + ( 1 - \alpha_\epsilon ) v^n \right)}{D} + \nu \nabla^2 v^n
\end{align}
Note that the linear Coriolis term, $f v$ is included in the non-linear term, $q H^v$, so we have a compensation term added that let's us treat some part of the Coriolis term implicitly in time.
The parameter $\alpha_f$ allows the Coriolis term to be centered in time ($\alpha=1/2$), preserving oscillation amplitudes, or made Euler-backward ($\alpha=1$) in time which damps oscillations without changing the balanced state.
Similarly, $\alpha_\epsilon$ allows the dissipation to be either explicit ($\alpha_\epsilon=0$) or Euler-backward ($\alpha_\epsilon=1$).

### Implicit-Coriolis and drag

Solution of the latter is managed by re-writing in terms of the velocity increment
\begin{align}
\Delta u &= u^{n+1} - u^n \\
\Delta v &= v^{n+1} - v^n
\end{align}
so the coupled problem becomes
\begin{align}
\left( \frac{1}{\Delta t} + \frac{ \alpha_\epsilon \epsilon }{D} \right) \Delta u - f \alpha_f \Delta v
&= \dot{u} \\
\left( \frac{1}{\Delta t} + \frac{ \alpha_\epsilon \epsilon }{D} \right) \Delta v + f \alpha_f \Delta u
&= \dot{v}
\end{align}
where the $\dot{u}$ and $\dot{v}$ are all accelerations evaluated explicitly:
\begin{align}
\dot{u}
& = \tilde{q}^{j} \overline{ H^v }^{ij} - \frac{1}{\Delta x} \delta_i \left( M^{n+1} + K^n \right)
+ \frac{\tau^x}{D} - \frac{\epsilon u^{n}}{D} + \nu \nabla^2 u^n \\
\dot{v}
& = - \tilde{q}^{i} \overline{ H^u }^{ij} - \frac{1}{\Delta y} \delta_j \left( M^{n+1} + K^n \right)
+ \frac{\tau^y}{D} - \frac{\epsilon v^{n}}{D} + \nu \nabla^2 v^n \\
\end{align}

Rearranging for $\Delta u$ and $\Delta v$, we get
\begin{align}
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta u
&= \Delta t^2 f \alpha_f \dot{v} + \Delta t \left( 1 + \frac{ \Delta t \alpha_\epsilon\epsilon }{D} \right)\dot{u} \\
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta v
&= - \Delta t^2  f \alpha_f \dot{u} + \Delta t \left( 1 + \frac{ \Delta t \alpha_\epsilon\epsilon }{D} \right)\dot{v} \\
\end{align}

Finally we apply the spatial discretization (interpolation) needed for the Coriolis terms:
\begin{align}
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta u
&= \Delta t \left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)\dot{u} + \Delta t f \alpha_f \overline{ \dot{v} }^{ij} \right) \\
\left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)^2 + \alpha_f^2 \Delta t^2  f^2 \right) \Delta v
&= \Delta t \left( \left( 1 + \frac{ \Delta t \alpha_\epsilon \epsilon }{D} \right)\dot{v} - \Delta t f \alpha_f \overline{ \dot{u} }^{ij} \right)
\end{align}